# Correcting the cell masks

The three annotated regions were segmented differently: the lung region holds cell
nuclei, while EMMA1 and X02 hold whole cells. This notebook brings them onto the same
footing by growing the nuclei and shrinking the whole cells, so that every region ends
up with a nucleus-sized object per cell.

Only the geometry changes. Each cell keeps its own positivity value (1 negative,
2 positive, 3 unknown) in every channel, and the geometry is the same for all channels,
exactly as in the originals.

The cells in all three volumes are separated by background, which is what lets a
connected component analysis pick them apart — and the outputs preserve that, so
`TrainModel.ipynb` keeps working.

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage

DATA_DIR = "TrainData"

EXPANSION_PX = 6    # how far to grow the lung nuclei
NUCLEUS_PX = 50     # an 8 um nucleus at 1 um/pixel covers about 50 pixels


def load_labels(filename):
    """Return the volume, one label per cell, and each cell's value per channel."""
    image = nib.load(f"{DATA_DIR}/{filename}")
    data = np.asanyarray(image.dataobj)

    cells, n_cells = ndimage.label(data[:, :, 0] > 0)

    # The cells are uniform within a channel, so one pixel each gives every value.
    representative = np.zeros(n_cells + 1, dtype=np.int64)
    representative[cells.ravel()] = np.arange(cells.size)
    values = data.reshape(-1, data.shape[2])[representative[1:]]

    return image, cells, values


def save_labels(image, cells, values, filename):
    """Paint the per channel values back onto the new geometry and write it out."""
    table = np.vstack([np.zeros((1, values.shape[1]), values.dtype), values])
    data = table[cells]

    out = nib.Nifti1Image(data, image.affine, image.header)
    out.set_data_dtype(data.dtype)
    nib.save(out, f"{DATA_DIR}/{filename}")

    kept = ndimage.label(data[:, :, 0] > 0)[1]
    sizes = np.bincount(cells.ravel())[1:]
    print(f"  wrote {filename}")
    print(f"  {kept} separate cells, median {np.median(sizes):.0f} px "
          f"(min {sizes.min()}, max {sizes.max()})")
    return data


def keep_largest_piece(cells):
    """Reduce every cell to its largest connected piece.

    Growing and shrinking can both break a cell into several pieces, and a connected
    component analysis would then count each piece as a separate cell.
    """
    pieces, n_pieces = ndimage.label(cells > 0)
    sizes = np.bincount(pieces.ravel(), minlength=n_pieces + 1)

    # Pieces never straddle two cells, so each one has a single owner.
    owner = np.zeros(n_pieces + 1, dtype=cells.dtype)
    owner[pieces.ravel()] = cells.ravel()

    largest = np.zeros(int(cells.max()) + 1, dtype=np.int64)
    for piece in np.argsort(sizes[1:]) + 1:   # smallest first, so the largest wins
        largest[owner[piece]] = piece

    return np.where(pieces == largest[cells], cells, 0)


def grow_cells(cells, distance):
    """Grow every cell outwards, stopping where cells meet so they never merge."""
    background = cells == 0
    depth, (y, x) = ndimage.distance_transform_edt(background, return_indices=True)

    added = background & (depth <= distance)
    grown = np.where(added, cells[y, x], cells)

    # Where two cells have grown into each other, drop the pixels that were added,
    # leaving a gap so a connected component analysis still separates them. Only
    # added pixels are dropped, so a cell can never be erased by a crowded neighbour.
    filled = np.where(grown == 0, grown.max() + 1, grown)
    crowded = ndimage.maximum_filter(grown, size=3) != ndimage.minimum_filter(filled, size=3)
    grown[crowded & added] = 0

    return keep_largest_piece(grown)


def shrink_cells(cells, target):
    """Erode every cell down to about `target` pixels, keeping its most central ones.

    Cells that are already at or below the target are left untouched.
    """
    depth = ndimage.distance_transform_edt(cells > 0)

    labels = cells.ravel()
    inside = np.flatnonzero(labels)

    # Group the pixels by cell, deepest first, and keep the first `target` of each.
    inside = inside[np.lexsort((-depth.ravel()[inside], labels[inside]))]
    grouped = labels[inside]
    rank = np.arange(inside.size) - np.searchsorted(grouped, grouped, side="left")

    shrunk = np.zeros_like(cells)
    keep = inside[rank < target]
    shrunk.ravel()[keep] = labels[keep]

    return keep_largest_piece(shrunk)


# ---------------------------------------------------------------------------
# 1. Lung: grow the nuclei into whole cells.
# ---------------------------------------------------------------------------
name = "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_corrected"

image, cells, values = load_labels(f"{name}.nii.gz")
before = np.bincount(cells.ravel())[1:]

grown = grow_cells(cells, EXPANSION_PX)

lost = np.setdiff1d(np.arange(1, len(before) + 1), np.unique(grown))
assert lost.size == 0, f"{lost.size} cells disappeared while growing"

print(f"{name[:38]}: {len(before)} nuclei, median {np.median(before):.0f} px "
      f"-> grown by {EXPANSION_PX} px")
save_labels(image, grown, values, f"{name}_dilated.nii.gz")

## EMMA1: shrink the whole cells to nuclei

In [ ]:
image, cells, values = load_labels("EMMA1_inference_corrected.nii")
before = np.bincount(cells.ravel())[1:]

shrunk = shrink_cells(cells, NUCLEUS_PX)

print(f"EMMA1: {len(before)} cells, median {np.median(before):.0f} px, "
      f"{(before <= NUCLEUS_PX).sum()} already <= {NUCLEUS_PX} px and left alone")
save_labels(image, shrunk, values, "EMMA1_inference_corrected_eroded.nii")

## X02: shrink the whole cells to nuclei

X02's cell mask lives in `X02_a0_full_mask_modified.nii.gz` — the file paired with
`X02_a0_full.nii.gz` in `TrainModel.ipynb`. `X02_a0_full.nii.gz` itself is the marker
stack and holds no cells.

In [ ]:
image, cells, values = load_labels("X02_a0_full_mask_modified.nii.gz")
before = np.bincount(cells.ravel())[1:]

shrunk = shrink_cells(cells, NUCLEUS_PX)

print(f"X02: {len(before)} cells, median {np.median(before):.0f} px, "
      f"{(before <= NUCLEUS_PX).sum()} already <= {NUCLEUS_PX} px and left alone")
save_labels(image, shrunk, values, "X02_a0_full_mask_modified_eroded.nii.gz")